In [7]:
# Qwen natively supports flex attention
# --------------------------------------

# Flex Attention with Qwen3 - Interactive Notebook Snippet
# Run this in your qwen.ipynb notebook

import torch
from torch.nn.attention.flex_attention import flex_attention, create_block_mask
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load the model (if not already loaded)
model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="cpu")

print(f"✓ Model loaded: {model_name}")
print(f"  Num heads: {model.config.num_attention_heads}")
print(f"  Hidden size: {model.config.hidden_size}")

# Test input
batch_size, seq_len = 2, 32
input_ids = torch.randint(0, tokenizer.vocab_size, (batch_size, seq_len))
memory_span = 16 

attention_mask = torch.ones_like(input_ids) # modify this

# modify from here, make it vectorize
for i in range(attention_mask.shape[0]):
    attention_mask[i, i-memory_span:i] = 0

print(f"\nInput shape: {input_ids.shape}")
print(f"Input range: [{input_ids.min().item()}, {input_ids.max().item()}]")


# === Test 1: Standard Attention ===
print("\n--- Standard Attention ---")
with torch.no_grad():
    standard_outputs = model(input_ids, attention_mask=attention_mask)
    standard_logits = standard_outputs.logits

print(f"✓ Standard logits shape: {standard_logits.shape}")
print(f"  Logits range: [{standard_logits.min().item():.4f}, {standard_logits.max().item():.4f}]")

# === Test 2: Flex Attention (Causal) ===
print("\n--- Flex Attention (Causal) ---")

def causal_mask_fn(b, h, q_idx, kv_idx):
    """Simple causal mask function"""
    return (q_idx >= kv_idx) & (q_idx - kv_idx <= memory_span)

# Create block mask
block_mask = create_block_mask(
    causal_mask_fn,
    B=batch_size,
    H=model.config.num_attention_heads,
    Q_LEN=seq_len,
    KV_LEN=seq_len,
    device=input_ids.device
)

print(f"✓ Block mask created: {type(block_mask)}")

# Use flex attention
with torch.no_grad():
    flex_outputs = model(input_ids, attention_mask=attention_mask, block_mask=block_mask)
    flex_logits = flex_outputs.logits

print(f"✓ Flex logits shape: {flex_logits.shape}")
print(f"  Logits range: [{flex_logits.min().item():.4f}, {flex_logits.max().item():.4f}]")

# === Compare Results ===
print("\n--- Comparison ---")
max_diff = torch.max(torch.abs(standard_logits - flex_logits)).item()
print(f"Max difference: {max_diff:.10f}")

if max_diff < 1e-5:
    print("🎉 PERFECT MATCH! Flex attention produces identical results!")
else:
    print(f"⚠️  Difference detected: {max_diff}")

# === Test 3: Custom SORL-style Mask ===
print("\n--- Custom SORL-style Mask ---")

def sorl_mask_fn(b, h, q_idx, kv_idx):
    """SORL-style mask with memory span constraint"""
    # Causal constraint
    causal = q_idx >= kv_idx
    
    # Document boundary (simplified - assumes 0 is BOS)
    docs = (input_ids[b] == 0).cumsum(0)
    same_doc = docs[q_idx] == docs[kv_idx]
    
    # Memory span constraint
    memory_span = 16  # Only attend to last 16 tokens
    recent = (q_idx - kv_idx) <= memory_span
    
    return causal & same_doc & recent

# Create SORL block mask
sorl_block_mask = create_block_mask(
    sorl_mask_fn,
    B=batch_size,
    H=model.config.num_attention_heads,
    Q_LEN=seq_len,
    KV_LEN=seq_len,
    device=input_ids.device
)

# Test SORL mask
with torch.no_grad():
    sorl_outputs = model(input_ids, block_mask=sorl_block_mask)
    sorl_logits = sorl_outputs.logits

print(f"✓ SORL logits shape: {sorl_logits.shape}")
print(f"  Logits range: [{sorl_logits.min().item():.4f}, {sorl_logits.max().item():.4f}]")

# === Memory Comparison ===
print("\n--- Memory Usage Comparison ---")
for test_seq_len in [512, 1024, 2048]:
    materialized_mb = (4 * test_seq_len * test_seq_len) / (1024**2)
    print(f"Seq {test_seq_len:4d}: Materialized {materialized_mb:6.1f}MB | Flex ~0MB | {materialized_mb:.0f}x savings")

print("\n🎯 Key Takeaway: Qwen3-0.6B natively supports flex attention!")
print("   Just pass block_mask to the model for instant memory savings!")


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✓ Model loaded: Qwen/Qwen3-0.6B
  Num heads: 16
  Hidden size: 1024

Input shape: torch.Size([2, 32])
Input range: [813, 150813]

--- Standard Attention ---
✓ Standard logits shape: torch.Size([2, 32, 151936])
  Logits range: [-14.0000, 19.7500]

--- Flex Attention (Causal) ---
✓ Block mask created: <class 'torch.nn.attention.flex_attention.BlockMask'>
✓ Flex logits shape: torch.Size([2, 32, 151936])
  Logits range: [-14.0000, 19.7500]

--- Comparison ---
Max difference: 0.0000000000
🎉 PERFECT MATCH! Flex attention produces identical results!

--- Custom SORL-style Mask ---
✓ SORL logits shape: torch.Size([2, 32, 151936])
  Logits range: [-14.0000, 19.7500]

--- Memory Usage Comparison ---
Seq  512: Materialized    1.0MB | Flex ~0MB | 1x savings
Seq 1024: Materialized    4.0MB | Flex ~0MB | 4x savings
Seq 2048: Materialized   16.0MB | Flex ~0MB | 16x savings

🎯 Key Takeaway: Qwen3-0.6B natively supports flex attention!
   Just pass block_mask to the model for instant memory savings!


In [8]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [1]:
from sorl.sorl_wrapper import SorlModelWrapper

# 1. We should not tie things, as per warning message, wte & lm_head have different weights? 
qwen_sorl = SorlModelWrapper.from_pretrained("Qwen/Qwen3-0.6B", 
                                             abstract_vocab_size_list=[128], 
                                             memory_span=64, 
                                             pad_token_id=0)

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [ ]:

# generate 
# parallel decode 
# compute loss


SorlModelWrapper(
  (model): Qwen3ForCausalLM(
    (model): Qwen3Model(
      (embed_tokens): Embedding(152065, 1024)
      (layers): ModuleList(
        (0-27): 28 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
            (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
            (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          )
          (mlp): Qwen3MLP(
            (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
            (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
            (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
            (act_fn): SiLUActivation()
          )
          (input_layerno